In [2]:
%pip install pymysql python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
# dotenv 불러오는 법
from dotenv import load_dotenv
load_dotenv()

# .env의 역할은 보안 정보들을 코드 외부에 관리하는 것
# 환경 변수 -> git에 올라가지 않게 관리 (.gitignore)

True

In [5]:
# 가져다 사용하는 법
import os
# os.environ['DB_HOST']
os.environ.get('DB_HOST', '테스트환경값')

'127.0.0.1'

In [6]:
type(os.environ) # dict처럼 사용

os._Environ

In [7]:
import pymysql

# mysql -u root -p
# mysql -h 127.0.0.1 -P 3306 -u root -ptest1234
conn = pymysql.connect(
    host=os.environ.get('DB_HOST', '127.0.0.1'), # 데이터베이스 서버 (컴퓨터) 주소 (localhost, 127. 0. 0. 1)
    port=int(os.environ.get('DB_PORT', '3306')),
    user=os.environ.get('DB_USER', 'analyst'),
    password=os.environ.get('DB_PASSWORD',''),
    database=os.environ.get('DB_NAME', 'shop_db'),
    charset='utf8mb4'
)
# bruta_force 완전탐색 공격: 데이터베이스 서버 있는 것 확인 -> 비밀번호 무작위 조합 시도

DB_CONFIG = {
    'host': os.environ.get('DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('DB_PORT', '3306')),
    'user': os.environ.get('DB_USER', 'analyst'),
    'password': os.environ.get('DB_PASSWORD',''),
    'database':os.environ.get('DB_NAME', 'shop_db'),
    'charset': 'utf8mb4',
    'autocommit': False,
}

from pymysql.cursors import DictCursor
conn = pymysql.connect(cursorclass=DictCursor, **DB_CONFIG)

In [8]:
# 가장 기본적인 SELECT 코드, tuple로 가져옴
with conn.cursor() as cur:
    cur.execute('SELECT * FROM tb_customer LIMIT 2')
    result = cur.fetchall()
result

[{'customer_id': 1,
  'customer_name': '홍길동',
  'city': '인천',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 1, 15),
  'grade': 'GOLD'},
 {'customer_id': 2,
  'customer_name': '이연걸',
  'city': '서울',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 2, 3),
  'grade': 'SILVER'}]

In [9]:
with conn.cursor() as cur:
    cur.execute('SELECT VERSION()')
    result = cur.fetchone()
result

{'VERSION()': '12.3.2-MariaDB'}

In [ ]:
# SQL Injection 공격
# SELECT * FROM tb_customer WHERE customer_name = '홍길동' OR 1=1;
sql = 'SELECT * FROM tb_customer WHERE customer_name = \'{}\''
with conn.cursor() as cur:
    name = '홍길동\' OR \'1\'=\'1'   
    cur.execute(sql.format(name))
    rows = cur.fetchall()
rows

[{'customer_id': 1,
  'customer_name': '홍길동',
  'city': '인천',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 1, 15),
  'grade': 'GOLD'},
 {'customer_id': 2,
  'customer_name': '이연걸',
  'city': '서울',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 2, 3),
  'grade': 'SILVER'},
 {'customer_id': 3,
  'customer_name': '이몽룡',
  'city': '대전',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 2, 20),
  'grade': 'BRONZE'},
 {'customer_id': 4,
  'customer_name': '성춘향',
  'city': '경기',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 3, 11),
  'grade': 'GOLD'},
 {'customer_id': 5,
  'customer_name': '김철수',
  'city': '부산',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 4, 5),
  'grade': 'BRONZE'},
 {'customer_id': 6,
  'customer_name': '박영희',
  'city': '서울',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 5, 19),
  'grade': 'SILVER'},
 {'customer_id': 7,
  'customer_name': 'Marie Dubois',
  'city': 'Paris',
  'country': 'FR',
  'signup_dt': datetime.date(2023, 6,

In [14]:
sql = 'SELECT * FROM tb_customer WHERE customer_name = \'{}\''
name = '홍길동\' OR \'1\'=\'1'
sql.format(name)

"SELECT * FROM tb_customer WHERE customer_name = '홍길동' OR '1'='1'"

In [16]:
sql = 'SELECT * FROM tb_customer WHERE customer_name IN (%s, %s)'
with conn.cursor() as cur:
    # name = '홍길동\' OR \'1\'=\'1'
    name = '홍길동'
    name2 = '이몽룡'
    cur.execute(sql, (name, name2 ))
    rows = cur.fetchall()
rows

[{'customer_id': 1,
  'customer_name': '홍길동',
  'city': '인천',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 1, 15),
  'grade': 'GOLD'},
 {'customer_id': 3,
  'customer_name': '이몽룡',
  'city': '대전',
  'country': 'KR',
  'signup_dt': datetime.date(2023, 2, 20),
  'grade': 'BRONZE'}]

### 실습 CRUD

In [34]:
# 코드 정리
import os
from dotenv import load_dotenv
import pymysql
from pymysql.cursors import DictCursor


load_dotenv()

DB_CONFIG = {
    'host': os.environ.get('DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('DB_PORT', '3306')),
    'user': os.environ.get('DB_USER', 'analyst'),
    'password': os.environ.get('DB_PASSWORD',''),
    'database':os.environ.get('DB_NAME', 'shop_db'),
    'charset': 'utf8mb4',
    'autocommit': False,
}

conn = pymysql.connect(cursorclass=DictCursor, **DB_CONFIG)


In [ ]:
# 01 테이블생성 id·name·email·phone·major

In [ ]:
sql = """
CREATE TABLE tb_student(
    id INT NOT NULL AUTO_INCREMENT,
    name VARCHAR(32) NOT NULL,
    email VARCHAR(255) NOT NULL,
    phone VARCHAR(32),
    major VARCHAR(32),
    PRIMARY KEY(id)
);
"""

with conn.cursor() as cur:
    cur.execute(sql) # INterfaceError (0, ) -> 연결 끊김 / 연결 후 재시도
    # conn.commit()
# conn.close()

In [ ]:
# 02 한건 INSERT lastrowid 확인

In [ ]:
sql = """
INSERT INTO tb_student(name, email, phone, major)
    VALUES (%s, %s, %s, %s);
"""
with conn.cursor() as cur:
    cur.execute(sql, ('홍길동', 'hong@gil.dong', '02-111-233', '성악과'))
    conn.commit()

In [ ]:
cur.execute('SELECT VERSION();')

In [21]:
cur.lastrowid

2

In [ ]:
# 03 네건 동시 INSERT executemany

In [23]:
sql = """
INSERT INTO tb_student(name, email, phone, major)
    VALUES (%s, %s, %s, %s);
"""
with conn.cursor() as cur:
    cur.executemany(sql, [
        ('이연걸', 'hong@gil.dong', '02-111-233', '성악과'),
        ('이몽룡', 'hong@gil.dong', '02-111-233', '성악과'),
        ('성춘향', 'hong@gil.dong', '02-111-233', '성악과'),
        ('김철수', 'hong@gil.dong', '02-111-233', '성악과'),
    ])
    conn.commit()

In [ ]:
# 04 전체조회 DictCursor로 받아 출력

In [24]:
sql = 'SELECT * FROM tb_student;'

with conn.cursor() as cur:
    cur.execute(sql)
    rows = cur.fetchall()
rows

[{'id': 1,
  'name': '홍길동',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'},
 {'id': 2,
  'name': '홍길동',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'},
 {'id': 3,
  'name': '이연걸',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'},
 {'id': 4,
  'name': '이몽룡',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'},
 {'id': 5,
  'name': '성춘향',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'},
 {'id': 6,
  'name': '김철수',
  'email': 'hong@gil.dong',
  'phone': '02-111-233',
  'major': '성악과'}]

In [ ]:
# 05 이메일 수정 파라미터 바인딩 사용

In [27]:
sql = """
UPDATE tb_student
SET email = %s
WHERE name = %s
"""

email = 'lee@mong.ry'
name = '이몽룡'

with conn.cursor() as cur:
    cur.execute(sql, (email,name))
    conn.commit()

In [28]:
cur.rowcount

0

In [ ]:
# 06 한명 삭제 rowcount 확인

In [29]:
sql = 'DELETE FROM tb_student WHERE name=%s'

name = '김철수'
with conn.cursor() as cur:
    cur.execute(sql, (name))
    conn.commit()

cur.rowcount

1

In [30]:
conn.close()

In [ ]:
"""
CREATE TABLE tb_account(
    acc_no VARCHAR(16) NOT NULL,
    balance INT NOT NULL,
    owner VARCHAR(32) NOT NULL,
    PRIMARY KEY (acc_no)
);
"""

"""
INSERT INTO tb_account(acc_no, owner, balance)
    VALUES ('A-001', '홍길동', 300000);
"""

In [35]:
with conn.cursor() as cur:
    cur.execute("UPDATE tb_account SET balance = 99999 where acc_no= 'A-001'")
    cur.execute("SELECT balance FROM tb_account where acc_no= 'A-001'")
    print(cur.fetchone()) # 99999로 보입니다
conn.close()

{'balance': 99999}


In [ ]:
import time
def _run(sql):
    # conn = get_connection()
    st = time.time()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)

        print('총 소요시간:', time.time() - st)

        conn.commit()
    except pymysql.err.OperationalError as e:
        print(f'[트랜잭션 에러]: {e}, 롤백합니다')
        conn.rollback()
    finally:
        conn.close()

_run('SELECT * FROM tb_student')

In [ ]:
import math
math.ceil(10003/1000)

In [ ]:
sql = 'INSERT INTO tb_account(acc_no, balance, owner) VALUES (%s, %s, %s)'
N = 10000
chunk_size = 1000

# 10003
# -> 11번

import time

st = time.time()
with conn.cursor() as cur:
    args = [(f'A-{str(i).zfill(5)}', 1000, '홍길동') for i in range(1, 10000)]
    # execute + commit 1만번
    for arg in args:
        cur.executemany(sql, args)
        conn.commit()
        
    # execute + commit 1번
    args = [(f'A-{str(i).zfill(5)}', 1000, '홍길동') for i in range(10001, 20000)]
    for arg in args:
        cur.executemany(sql, args)
    conn.commit()

    # excute_many
    args = [(f'A-{str(i).zfill(5)}', 1000, '홍길동') for i in range(20001, 30000)]
    cur.executemany(sql, args)
    conn.commit()

    # 청크
    args = [(f'A-{str(i).zfill(5)}', 1000, '홍길동') for i in range(30001, 40000)]
    for i in range(N//chunk_size):
        cur.executemany(sql, args[i*chunk_size:(i+1)*chunk_size])
        conn.commit()
        
'총 소요시간' + str(time.time() - st)

# 5.073619604110718 execute + commit
# 0.716788291931152 execute + 1회 commit
# 0.063856601715087 executemany + commit
# 0.078795909881591 청크 executemany + commit

In [ ]:
table = 'tb_customer'
cols = ['customer_name', 'email']
values = ('홍길동', 'hong@gil.dong')

sql = f"""
    INSERT INTO {table}({','.join(cols)})
        VALUES ({','.join(['%s']*len(values))})
"""
print(sql)

_run(sql, args=values)

In [ ]:
','.join(['%s']*len(values))

In [ ]:
table = 'tb_student'
cols = ['*']
sql = f"""
        SELECT {','.join(cols)}
        FROM {table}
        """

In [ ]:
print(sql)

In [ ]:
from db import MariaDBHandler
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    'host': os.environ.get('DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('DB_PORT', '3306')),
    'user': os.environ.get('DB_USER', 'analyst'),
    'password': os.environ.get('DB_PASSWORD', ''),
    'database': os.environ.get('DB_NAME', 'shop_db'),
}

db = MariaDBHandler(**DB_CONFIG)

with db:
    result = db.select('tb_customer', ['country'])
result

In [ ]:
import pandas as pd

df = pd.DataFrame(result)

In [ ]:
country_df = pd.DataFrame(df.value_counts('country'))
country_df['비중'] = country_df / country_df.sum()

In [ ]:
country_df

In [ ]:
import logging
import sys
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s | %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('report.log', encoding='utf-8')
    ])


logging.info('[RAW] %d건 적재 완료', 30)

In [ ]:
log = logging.getLogger('Pipeline')

log.info('[RAW] 데이터 %d건 적재 완료', 30)

In [ ]:
log.warning('워닝')

In [ ]:
log.error('오류 발생')